<a href="https://colab.research.google.com/github/aayushhh-operator/Crime-Forecasting/blob/Aayush/GNN%2BLSTM(added_layers).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install folium spektral scikit-learn networkx

In [ ]:
import pandas as pd
import numpy as np
import folium
import networkx as nx
from datetime import datetime

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

from spektral.data import Graph
from spektral.layers import GCNConv
from spektral.data import Dataset, Loader
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, LSTM, Input, Concatenate, TimeDistributed, GlobalAveragePooling1D
from tensorflow.keras.optimizers import Adam

In [ ]:
from tensorflow.keras.layers import Dropout, Concatenate, BatchNormalization
from tensorflow.keras.regularizers import l2

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/GDG/Research Paper/Datasets/Clean_LA_Crime.csv')

In [ ]:
df.head()

,DATE OCC,Day,Month,Year,Day of Week,Is_Weekend,Is_Holiday,TIME OCC,Hour,Minute,Time of Day,AREA,AREA NAME,Crm Cd,Crm Cd Desc,LOCATION,LAT,LON
0,2020-03-01,1,3,2020,Sunday,1,0,21:30:00,21,30,Night,7,Wilshire,0,Vehicle Theft,1900 S LONGWOOD AV,34.0375,-118.3506
1,2020-02-08,8,2,2020,Saturday,1,0,18:00:00,18,0,Evening,1,Central,1,Burglary,1000 S FLOWER ST,34.0444,-118.2628
2,2020-11-04,4,11,2020,Wednesday,0,0,17:00:00,17,0,Evening,3,Southwest,0,Vehicle Theft,1400 W 37TH ST,34.0210,-118.3002
3,2020-03-10,10,3,2020,Tuesday,0,0,20:37:00,20,37,Evening,9,Van Nuys,2,Theft,14000 RIVERSIDE DR,34.1576,-118.4387
4,2020-09-09,9,9,2020,Wednesday,0,0,06:30:00,6,30,Early Morning,4,Hollenbeck,0,Vehicle Theft,200 E AVENUE 28,34.0820,-118.2130


In [ ]:
df['DATETIME'] = pd.to_datetime(df['DATE OCC'].astype(str) + ' ' + df['TIME OCC'].astype(str))

In [ ]:
df['Day of Week'] = df['Day of Week'].map({
    'Monday': 0,
    'Tuesday': 1,
    'Wednesday': 2,
    'Thursday': 3,
    'Friday': 4,
    'Saturday': 5,
    'Sunday': 6
})

In [ ]:
df['Time of Day'].unique()

array(['Night', 'Evening', 'Early Morning', 'Afternoon', 'Late Night',
       'Morning'], dtype=object)

In [ ]:
df['Time of Day'] = df['Time of Day'].map({
    'Early Morning': 0,
    'Morning': 1,
    'Afternoon': 2,
    'Evening': 3,
    'Night': 4,
    'Late Night': 5
})

In [ ]:
df.head()

,DATE OCC,Day,Month,Year,Day of Week,Is_Weekend,Is_Holiday,TIME OCC,Hour,Minute,Time of Day,AREA,AREA NAME,Crm Cd,Crm Cd Desc,LOCATION,LAT,LON,DATETIME
0,2020-03-01,1,3,2020,6,1,0,21:30:00,21,30,4,7,Wilshire,0,Vehicle Theft,1900 S LONGWOOD AV,34.0375,-118.3506,2020-03-01 21:30:00
1,2020-02-08,8,2,2020,5,1,0,18:00:00,18,0,3,1,Central,1,Burglary,1000 S FLOWER ST,34.0444,-118.2628,2020-02-08 18:00:00
2,2020-11-04,4,11,2020,2,0,0,17:00:00,17,0,3,3,Southwest,0,Vehicle Theft,1400 W 37TH ST,34.0210,-118.3002,2020-11-04 17:00:00
3,2020-03-10,10,3,2020,1,0,0,20:37:00,20,37,3,9,Van Nuys,2,Theft,14000 RIVERSIDE DR,34.1576,-118.4387,2020-03-10 20:37:00
4,2020-09-09,9,9,2020,2,0,0,06:30:00,6,30,0,4,Hollenbeck,0,Vehicle Theft,200 E AVENUE 28,34.0820,-118.2130,2020-09-09 06:30:00


In [ ]:
features = ['DATETIME', 'Day', 'Month', 'Year', 'Day of Week', 'Hour', 'Minute', 'Is_Weekend', 'Is_Holiday', 'Time of Day', 'AREA', 'Crm Cd', 'LAT', 'LON']

In [ ]:
df = df[features]

In [ ]:
df.head()

,DATETIME,Day,Month,Year,Day of Week,Hour,Minute,Is_Weekend,Is_Holiday,Time of Day,AREA,Crm Cd,LAT,LON
0,2020-03-01 21:30:00,1,3,2020,6,21,30,1,0,4,7,0,34.0375,-118.3506
1,2020-02-08 18:00:00,8,2,2020,5,18,0,1,0,3,1,1,34.0444,-118.2628
2,2020-11-04 17:00:00,4,11,2020,2,17,0,0,0,3,3,0,34.0210,-118.3002
3,2020-03-10 20:37:00,10,3,2020,1,20,37,0,0,3,9,2,34.1576,-118.4387
4,2020-09-09 06:30:00,9,9,2020,2,6,30,0,0,0,4,0,34.0820,-118.2130


In [ ]:
scaler_features = MinMaxScaler()
X = scaler_features.fit_transform(df[['Minute', 'Hour', 'Day', 'Month', 'Year', 'Day of Week', 'Time of Day', 'Crm Cd', 'AREA']])

In [ ]:
X = pd.DataFrame(X, columns=['Minute', 'Hour', 'Day', 'Month', 'Year', 'Day of Week', 'Time of Day', 'Crm Cd', 'AREA'])
X.head()

,Minute,Hour,Day,Month,Year,Day of Week,Time of Day,Crm Cd,AREA
0,0.508475,0.913043,0.000000,0.181818,0.0,1.000000,0.8,0.0000,0.30
1,0.000000,0.782609,0.233333,0.090909,0.0,0.833333,0.6,0.0625,0.00
2,0.000000,0.739130,0.100000,0.909091,0.0,0.333333,0.6,0.0000,0.10
3,0.627119,0.869565,0.300000,0.181818,0.0,0.166667,0.6,0.1250,0.40
4,0.508475,0.260870,0.266667,0.727273,0.0,0.333333,0.0,0.0000,0.15


In [ ]:
scaler_coords = MinMaxScaler()
Y = scaler_coords.fit_transform(df[['LAT', 'LON']])

In [ ]:
Y = pd.DataFrame(Y, columns=['LAT', 'LON'])
Y.head()

,LAT,LON
0,0.991356,0.002671
1,0.991557,0.003411
2,0.990875,0.003096
3,0.994854,0.001929
4,0.992652,0.003831


In [ ]:
data = np.hstack([X, Y])

In [ ]:
SEQ_LEN = 20

In [ ]:
def create_sequences(data, seq_len=SEQ_LEN):
    X_seq, Y_seq = [], []
    for i in range(len(data) - seq_len):
        X_seq.append(data[i:i+seq_len, :-2])  # features
        Y_seq.append(data[i+seq_len, -2:])    # next location (LAT, LON)
    return np.array(X_seq), np.array(Y_seq)

X_seq, Y_seq = create_sequences(data, seq_len=SEQ_LEN)

In [ ]:
unique_areas = df['AREA'].unique()
G_nx = nx.complete_graph(len(unique_areas))
adj = nx.to_numpy_array(G_nx)

In [ ]:
N_areas = len(unique_areas)
num_features = X_seq.shape[2]

area_input = Input(shape=(SEQ_LEN, num_features), name="lstm_input")
x = LSTM(128, return_sequences=True)(area_input)
x = LSTM(64)(x)  # use final hidden state

In [ ]:
gnn_input = Input(shape=(num_features,), name="gnn_input")
gnn_output = Dense(64, activation='relu', kernel_regularizer=l2(0.001))(gnn_input)
gnn_output = Dense(32, activation='relu')(gnn_output)

In [ ]:
merged = Concatenate()([x, gnn_output])
dense = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(merged)
dense = BatchNormalization()(dense)
dense = Dropout(0.3)(dense)
dense = Dense(64, activation='relu')(dense)
dense = Dropout(0.3)(dense)

In [ ]:
out = Dense(2, activation='linear')(dense)

In [ ]:
model = Model(inputs=[area_input, gnn_input], outputs=out)
model.compile(optimizer=Adam(learning_rate=0.0005), loss='mse')

In [ ]:
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ lstm_input          │ (None, 20, 9)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gnn_input           │ (None, 9)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 20, 128)   │     70,656 │ lstm_input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │        640 │ gnn_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 64)        │     49,408 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 32)        │      2,080 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 96)        │          0 │ lstm_2[0][0],     │
│ (Concatenate)       │                   │            │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 128)       │     12,416 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 128)       │        512 │ dense_5[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 64)        │      8,256 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 2)         │        130 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 144,098 (562.88 KB)

 Trainable params: 143,842 (561.88 KB)

 Non-trainable params: 256 (1.00 KB)

In [ ]:
area_features = np.eye(N_areas)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_seq, Y_seq, test_size=0.3, random_state=42)

In [ ]:
gnn_inputs = X_train[:, -1] # Extract the 'AREA' column which is the last column

In [ ]:
model.fit([X_train, gnn_inputs], y_train, validation_split=0.1, epochs=10, batch_size=32)

Epoch 1/10
19784/19784 ━━━━━━━━━━━━━━━━━━━━ 480s 24ms/step - loss: 0.0522 - val_loss: 0.0023
Epoch 2/10
19784/19784 ━━━━━━━━━━━━━━━━━━━━ 471s 24ms/step - loss: 0.0023 - val_loss: 0.0023
Epoch 3/10
19784/19784 ━━━━━━━━━━━━━━━━━━━━ 453s 23ms/step - loss: 0.0022 - val_loss: 0.0023
Epoch 4/10
19784/19784 ━━━━━━━━━━━━━━━━━━━━ 472s 24ms/step - loss: 0.0021 - val_loss: 0.0023
Epoch 5/10
19784/19784 ━━━━━━━━━━━━━━━━━━━━ 458s 23ms/step - loss: 0.0023 - val_loss: 0.0023
Epoch 6/10
19784/19784 ━━━━━━━━━━━━━━━━━━━━ 449s 23ms/step - loss: 0.0021 - val_loss: 0.0023
Epoch 7/10
19784/19784 ━━━━━━━━━━━━━━━━━━━━ 461s 23ms/step - loss: 0.0023 - val_loss: 0.0023
Epoch 8/10
19784/19784 ━━━━━━━━━━━━━━━━━━━━ 470s 24ms/step - loss: 0.0022 - val_loss: 0.0023
Epoch 9/10
19784/19784 ━━━━━━━━━━━━━━━━━━━━ 467s 24ms/step - loss: 0.0022 - val_loss: 0.0023
Epoch 10/10
19784/19784 ━━━━━━━━━━━━━━━━━━━━ 469s 24ms/step - loss: 0.0022 - val_loss: 0.0023


In [ ]:
gnn_test_inputs = X_test[:, -1] # Extract the 'AREA' column which is the last column
pred_coords = model.predict([X_test, gnn_test_inputs])

9421/9421 ━━━━━━━━━━━━━━━━━━━━ 89s 9ms/step


In [ ]:
pred_coords_real = scaler_coords.inverse_transform(pred_coords)
true_coords_real = scaler_coords.inverse_transform(y_test)

In [ ]:
m = folium.Map(location=[34.05, -118.25], zoom_start=10)

In [ ]:
for i in range(10):
    pred_lat, pred_lon = pred_coords_real[i]
    true_lat, true_lon = true_coords_real[i]

    folium.Marker(location=[true_lat, true_lon], popup='True', icon=folium.Icon(color='green')).add_to(m)
    folium.Marker(location=[pred_lat, pred_lon], popup='Predicted', icon=folium.Icon(color='red')).add_to(m)
    folium.PolyLine(locations=[[true_lat, true_lon], [pred_lat, pred_lon]], color='blue').add_to(m)

In [ ]:
m

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
pred_coords_real = scaler_coords.inverse_transform(pred_coords)
true_coords_real = scaler_coords.inverse_transform(y_test)

In [ ]:
rmse_lat = np.sqrt(mean_squared_error(true_coords_real[:, 0], pred_coords_real[:, 0]))
rmse_lon = np.sqrt(mean_squared_error(true_coords_real[:, 1], pred_coords_real[:, 1]))

In [ ]:
rmse_total = np.sqrt(np.mean((true_coords_real - pred_coords_real)**2))

In [ ]:
print(f"RMSE (Latitude): {rmse_lat:.5f}")
print(f"RMSE (Longitude): {rmse_lon:.5f}")
print(f"Combined RMSE: {rmse_total:.5f}")

RMSE (Latitude): 0.05335
RMSE (Longitude): 0.04418
Combined RMSE: 0.04898
